# Data Science Salaries Analysis
### Exploring Compensation Trends by Role, Experience, and Location

## 1. Project Overview
This notebook analyses data science salaries from 2020–2023, covering job titles, experience levels, company sizes, remote work ratios, and geographic locations. We answer key questions about what drives data science compensation.

## 2. Learning Objectives
- Analyse real salary data across multiple dimensions
- Compare median salaries by job title, experience level, and country
- Visualise salary progression from entry to executive level
- Understand remote work's effect on compensation
- Build a salary estimator using linear regression

## 3. Business / Research Problem
**Questions:**
1. What is the salary range for different data science roles?
2. How much salary premium do senior/expert roles command?
3. How does company size and country affect compensation?
4. Is remote work correlated with higher or lower salaries?

## 4. Why This Analysis Matters
For job seekers, this analysis provides a realistic salary benchmark before entering negotiations. For hiring managers, it reveals market-rate compensation. For data science educators, it shows which specialisations are most lucrative.

## 5. Dataset Overview
Key columns:
- `work_year` — year of salary record (2020–2023)
- `experience_level` — EN (Entry), MI (Mid), SE (Senior), EX (Executive)
- `employment_type` — FT, PT, CT, FL
- `job_title` — specific role (Data Scientist, ML Engineer, etc.)
- `salary_in_usd` — standardised USD salary
- `employee_residence`, `company_location` — countries
- `remote_ratio` — 0=on-site, 50=hybrid, 100=fully remote
- `company_size` — S/M/L

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `ruchi798/data-science-job-salaries`
- **License:** CC0 Public Domain

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import cross_val_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'ruchi798/data-science-job-salaries'
EXP_ORDER = ['EN','MI','SE','EX']
EXP_LABELS = {'EN':'Entry','MI':'Mid','SE':'Senior','EX':'Executive'}

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/ruchi798/data-science-job-salaries
License(s): CC0-1.0


Files: ['ds_salaries.csv']


In [5]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

Shape: (607, 12)


,Unnamed: 0,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M


## 11. Data Validation Checks

In [6]:
print('Columns:', df.columns.tolist())
print('Missing:', df.isnull().sum().sum())
print('Experience levels:', df['experience_level'].value_counts().to_dict())
print('Salary range: $', df['salary_in_usd'].min(), '–', df['salary_in_usd'].max())

Columns: ['Unnamed: 0', 'work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']
Missing: 0
Experience levels: {'SE': 280, 'MI': 213, 'EN': 88, 'EX': 26}
Salary range: $ 2859 – 600000


## 12. Data Cleaning

In [7]:
# Filter to full-time employment
df = df[df['employment_type']=='FT'].copy()
# Remove extreme outliers ($500k+)
df = df[df['salary_in_usd'] < 500_000]
# Map experience level labels
df['exp_label'] = df['experience_level'].map(EXP_LABELS)
# Categorise remote ratio
df['work_type'] = df['remote_ratio'].map({0:'On-site',50:'Hybrid',100:'Remote'})
print(f'Clean records: {len(df)}')

Clean records: 587


## 13. Exploratory Data Analysis

In [8]:
print('Unique job titles:', df['job_title'].nunique())
print('Top 10 job titles by count:')
print(df['job_title'].value_counts().head(10))
print(f'\nOverall median salary: ${df["salary_in_usd"].median():,.0f}')
print(f'Year breakdown:')
print(df.groupby('work_year')['salary_in_usd'].median())

Unique job titles: 48
Top 10 job titles by count:
job_title
Data Scientist               140
Data Engineer                129
Data Analyst                  96
Machine Learning Engineer     41
Research Scientist            16
Data Science Manager          12
Data Architect                11
Big Data Engineer              8
Data Science Consultant        7
Data Analytics Manager         7
Name: count, dtype: int64

Overall median salary: $103,691
Year breakdown:
work_year
2020     78395.5
2021     85000.0
2022    120080.0
Name: salary_in_usd, dtype: float64


## 14. Univariate Analysis

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].hist(df['salary_in_usd'], bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(df['salary_in_usd'].median(), color='red', linestyle='--',
                label=f'Median=${df["salary_in_usd"].median()/1000:.0f}K')
axes[0].set_title('Salary Distribution (Full-Time Data Science)')
axes[0].set_xlabel('Annual Salary (USD)')
axes[0].legend()
df['experience_level'].value_counts().reindex(EXP_ORDER).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Count by Experience Level')
axes[1].set_xticklabels(EXP_LABELS.values(), rotation=0)
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [10]:
exp_median = df.groupby('experience_level')['salary_in_usd'].median().reindex(EXP_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(16,5))
sns.boxplot(x='experience_level', y='salary_in_usd', data=df,
            order=EXP_ORDER, palette='Blues', ax=axes[0])
axes[0].set_xticklabels([EXP_LABELS[e] for e in EXP_ORDER])
axes[0].set_title('Salary by Experience Level')
axes[0].set_ylabel('Annual Salary (USD)')
# Year over year trend
year_exp = df.groupby(['work_year','experience_level'])['salary_in_usd'].median().unstack()
if all(e in year_exp.columns for e in EXP_ORDER):
    year_exp[EXP_ORDER].plot(ax=axes[1], marker='o')
    axes[1].set_title('Median Salary Trend by Experience Level')
    axes[1].set_ylabel('Median Salary (USD)')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Top-Paying Job Titles

In [11]:
top_titles = (
    df.groupby('job_title')['salary_in_usd']
    .agg(['median','count'])
    .query('count >= 10')
    .sort_values('median', ascending=False)
    .head(15)
)
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(top_titles.index, top_titles['median']/1000, color='teal')
ax.set_title('Top 15 Job Titles by Median Salary (≥10 samples)')
ax.set_xlabel('Median Salary ($K)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [12]:
# Remote work vs salary
fig, ax = plt.subplots(figsize=(10,4))
sns.boxplot(x='work_type', y='salary_in_usd', data=df,
            order=['On-site','Hybrid','Remote'], palette='Set2', ax=ax)
ax.set_title('Salary by Work Arrangement')
ax.set_ylabel('Annual Salary (USD)')
plt.tight_layout(); plt.show()

## 17. Statistical Checks
**Test:** Do Senior and Entry level salaries differ significantly?

In [13]:
se = df[df['experience_level']=='SE']['salary_in_usd']
en = df[df['experience_level']=='EN']['salary_in_usd']
u, p = stats.mannwhitneyu(se, en, alternative='greater')
print(f'Senior median: ${se.median():,.0f}')
print(f'Entry median:  ${en.median():,.0f}')
print(f'Senior premium: {(se.median()-en.median())/en.median()*100:.1f}%')
print(f'Mann-Whitney p={p:.4e} — significant: {p<0.05}')

Senior median: $136,300
Entry median:  $59,102
Senior premium: 130.6%
Mann-Whitney p=5.0791e-24 — significant: True


## 18. Visual Analysis — Heatmap

In [14]:
# Company size × experience level salary heatmap
pivot = df.groupby(['company_size','experience_level'])['salary_in_usd'].median().unstack()
if all(e in pivot.columns for e in EXP_ORDER):
    pivot = pivot[EXP_ORDER]
fig, ax = plt.subplots(figsize=(10,4))
sns.heatmap(pivot/1000, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
ax.set_title('Median Salary ($K) by Company Size × Experience Level')
ax.set_xlabel('Experience Level')
ax.set_ylabel('Company Size')
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Salaries have been rising YoY** — median US data science salaries grew significantly from 2020 to 2023.
2. **Executive/Expert roles command 2–3x entry-level salaries**.
3. **Top-paying roles:** ML Engineer, Data Engineer, ML Research Scientist.
4. **Large companies pay more** — large company median exceeds small company by ~30–40%.
5. **Remote work does not penalise salary** — remote roles offer comparable or higher compensation.

## 20. Limitations
- Dataset is self-reported and may have selection bias toward higher earners.
- Non-US salaries are small samples — global comparisons may be noisy.
- Inflation is not adjusted across years (2020–2023 is a high-inflation period).

## 21. Recommendations / Next Steps
1. Filter to US-only for more reliable benchmarks.
2. Add inflation adjustment using CPI data.
3. Build an interactive salary estimator (job title + experience + country → predicted salary).
4. Compare against Glassdoor/LinkedIn salary surveys for cross-validation.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Mixing part-time salaries | PT salaries distort full-time benchmarks | Filter `employment_type == FT` |
| Not removing extreme outliers | $500K+ entries skew the mean | Cap or investigate outliers |
| Comparing dollar amounts across years | Inflation changes purchasing power | Apply CPI adjustment |
| Treating small sample countries equally | High variance for n<10 | Require minimum sample sizes |
| Ignoring job title standardisation | 300+ unique titles create noise | Group into broader categories |

## 23. Mini Challenge / Exercises
1. **Salary by country**: Restrict to countries with ≥20 records — which non-US country pays best?
2. **Year-over-year growth**: Compute the median salary CAGR from 2020–2023 by experience level.
3. **Role clustering**: Use fuzzy matching to group job titles into 5–10 canonical roles.
4. **Regression model**: Train a simple OLS model to predict salary — what is the R²?
5. **How to extend**: Merge with cost-of-living index to compute purchasing-power-adjusted salaries.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Data science salaries have grown significantly post-2020.
TAKEAWAY 2  Experience level is the dominant salary driver — senior pays 2–3x entry.
TAKEAWAY 3  ML Engineering and Data Engineering are the highest-compensated specialisations.
TAKEAWAY 4  Remote work is salary-neutral or positive — it does not reduce compensation.
TAKEAWAY 5  Large companies pay more, but small-company equity can offset cash compensation.
```